# CNN evaluation HDF5 demo

This notebook evaluates CNN prediction files generated by:

```bash
python cbc_pe/scripts/train_cnn_hdf5.py --config <config.json>
```

The current goal is to compare candidate CNN architectures trained on the 100k HDF5 dataset using the 80/20 train/validation split.

This notebook focuses on:

- global validation metrics
- per-label validation metrics
- standardized-space metrics
- physical-space metrics
- absolute-error quantiles
- detailed diagnostics for a selected model

In [4]:
from pathlib import Path
import os
import sys
import json

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()

# Allow running either from repository root or from cbc_pe/.
if PROJECT_ROOT.name != "cbc_pe" and (PROJECT_ROOT / "cbc_pe").exists():
    PROJECT_ROOT = PROJECT_ROOT / "cbc_pe"

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Main data root for CIEMAT/local office workflow.
DATA_ROOT = Path("/data/vserrano/cbc_pe_data")

DATA_PROCESSED = DATA_ROOT / "processed"
DATA_RESULTS = DATA_ROOT / "results"
DATA_MODELS = DATA_ROOT / "models"

dataset_id = "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
MODEL_RESULTS_DIR = DATA_RESULTS / dataset_id

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("DATA_PROCESSED:", DATA_PROCESSED)
print("MODEL_RESULTS_DIR:", MODEL_RESULTS_DIR)
print("MODEL_RESULTS_DIR exists:", MODEL_RESULTS_DIR.exists())

PROJECT_ROOT: /afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe/notebooks
DATA_ROOT: /data/vserrano/cbc_pe_data
DATA_PROCESSED: /data/vserrano/cbc_pe_data/processed
MODEL_RESULTS_DIR: /data/vserrano/cbc_pe_data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n100_000
MODEL_RESULTS_DIR exists: True


## Register models to compare

Add one entry per trained model.

When a new model finishes training, add its prediction file to `prediction_files` and re-run the comparison cells.

In [5]:
prediction_files = {
    "M00_baseline_emb64_seed123": MODEL_RESULTS_DIR / (
        "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
        "_SimpleCNN_Baseline_M00_simple_emb64_mse_MSELoss_seed123"
        "_train_val_predictions_embeddings.npz"
    ),
    "M00_baseline_emb64_seed124": MODEL_RESULTS_DIR / (
        "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
        "_SimpleCNN_Baseline_M00_simple_emb64_mse_MSELoss_seed124"
        "_train_val_predictions_embeddings.npz"
    ),
    "M00_baseline_emb64_seed125": MODEL_RESULTS_DIR / (
        "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
        "_SimpleCNN_Baseline_M00_simple_emb64_mse_MSELoss_seed125"
        "_train_val_predictions_embeddings.npz"
    ),
    "M01_pool1_emb128": MODEL_RESULTS_DIR / (
        "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
        "_SimpleCNN_Pool_M01_emb128_pool1_MSELoss_seed123"
        "_train_val_predictions_embeddings.npz"
    ),
    "M02_pool4_emb128": MODEL_RESULTS_DIR / (
        "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
        "_SimpleCNN_Pool_M02_emb128_pool4_MSELoss_seed123"
        "_train_val_predictions_embeddings.npz"
    ),
    "M04_pooldeep_emb128_pool4": MODEL_RESULTS_DIR / (
        "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
        "_SimpleCNN_PoolDeep_M04_emb128_pool4_deephead_MSELoss_seed123"
        "_train_val_predictions_embeddings.npz"
    ),
    "M04_pooldeep_emb128_pool4_drop_wd": MODEL_RESULTS_DIR / (
        "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
        "_SimpleCNN_PoolDeep_M04_emb128_pool4_deephead_dropout020_wd1e3_MSELoss_seed123"
        "_train_val_predictions_embeddings.npz"
    ),
    "M06_WideCNN_emb128_pool4": MODEL_RESULTS_DIR / (
        "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
        "_WideCNN_Pool_M06_emb128_pool4_widecnn_MSELoss_seed123"
        "_train_val_predictions_embeddings.npz"
    ),
}



for model_id, path in prediction_files.items():
    print(f"{model_id:30s} exists={path.exists()}  file={path.name}")

M00_baseline_emb64_seed123     exists=True  file=bbh_processed_4s_seobnrv4opt_snr10-25_n100_000_SimpleCNN_Baseline_M00_simple_emb64_mse_MSELoss_seed123_train_val_predictions_embeddings.npz
M00_baseline_emb64_seed124     exists=True  file=bbh_processed_4s_seobnrv4opt_snr10-25_n100_000_SimpleCNN_Baseline_M00_simple_emb64_mse_MSELoss_seed124_train_val_predictions_embeddings.npz
M00_baseline_emb64_seed125     exists=True  file=bbh_processed_4s_seobnrv4opt_snr10-25_n100_000_SimpleCNN_Baseline_M00_simple_emb64_mse_MSELoss_seed125_train_val_predictions_embeddings.npz
M01_pool1_emb128               exists=True  file=bbh_processed_4s_seobnrv4opt_snr10-25_n100_000_SimpleCNN_Pool_M01_emb128_pool1_MSELoss_seed123_train_val_predictions_embeddings.npz
M02_pool4_emb128               exists=True  file=bbh_processed_4s_seobnrv4opt_snr10-25_n100_000_SimpleCNN_Pool_M02_emb128_pool4_MSELoss_seed123_train_val_predictions_embeddings.npz
M04_pooldeep_emb128_pool4      exists=True  file=bbh_processed_4s_seobn

## Metric helper functions

In [6]:
def regression_metrics(y_true, y_pred):
    residual = y_true - y_pred

    mse = np.mean(residual**2, axis=0)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(residual), axis=0)
    bias = np.mean(residual, axis=0)
    residual_std = np.std(residual, axis=0)

    ss_res = np.sum(residual**2, axis=0)
    ss_tot = np.sum((y_true - np.mean(y_true, axis=0))**2, axis=0)
    r2 = 1.0 - ss_res / ss_tot

    global_mse = np.mean(residual**2)
    global_rmse = np.sqrt(global_mse)
    global_mae = np.mean(np.abs(residual))

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "bias": bias,
        "residual_std": residual_std,
        "r2": r2,
        "global_mse": global_mse,
        "global_rmse": global_rmse,
        "global_mae": global_mae,
    }


def abs_error_quantiles(y_true, y_pred, label_names, model_id, space):
    abs_err = np.abs(y_true - y_pred)
    rows = []

    for j, label in enumerate(label_names):
        q50, q90, q95, q99 = np.quantile(abs_err[:, j], [0.50, 0.90, 0.95, 0.99])

        rows.append({
            "model_id": model_id,
            "space": space,
            "label": label,
            "q50_abs_error": q50,
            "q90_abs_error": q90,
            "q95_abs_error": q95,
            "q99_abs_error": q99,
            "max_abs_error": abs_err[:, j].max(),
        })

    return rows


def get_label_names(data):
    if "label_names" in data.files:
        return [str(x) for x in data["label_names"].tolist()]
    return ["chirp_mass", "total_mass", "chi_eff"]


def load_history_npz(history_path):
    history_path = Path(history_path)

    if not history_path.exists():
        return None

    with np.load(history_path, allow_pickle=True) as data:
        files = list(data.files)

        if "train_loss" in files and "val_loss" in files:
            arrays = {
                key: data[key]
                for key in files
                if np.asarray(data[key]).ndim <= 1
            }
            history_df = pd.DataFrame(arrays)

        elif "history" in files:
            obj = data["history"].tolist()

            if isinstance(obj, dict):
                history_df = pd.DataFrame(obj)
            elif isinstance(obj, list):
                history_df = pd.DataFrame(obj)
            else:
                raise ValueError(f"Unsupported history object in {history_path}")

        else:
            arrays = {
                key: data[key]
                for key in files
                if np.asarray(data[key]).ndim <= 1
            }
            history_df = pd.DataFrame(arrays)

    history_df = history_df.rename(columns={
        "train": "train_loss",
        "val": "val_loss",
        "valid_loss": "val_loss",
        "validation_loss": "val_loss",
    })

    if "epoch" not in history_df.columns:
        history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))

    return history_df


def summarize_history(history_df):
    h = history_df.copy()

    required = {"epoch", "train_loss", "val_loss"}
    missing = required - set(h.columns)

    if missing:
        raise ValueError(
            f"Missing columns {missing}. Available columns: {h.columns.tolist()}"
        )

    h["train_loss"] = h["train_loss"].astype(float)
    h["val_loss"] = h["val_loss"].astype(float)

    best_idx = h["val_loss"].idxmin()

    best_epoch = int(h.loc[best_idx, "epoch"])
    stop_epoch = int(h["epoch"].iloc[-1])

    best_val_loss = float(h.loc[best_idx, "val_loss"])
    train_loss_at_best = float(h.loc[best_idx, "train_loss"])

    final_train_loss = float(h["train_loss"].iloc[-1])
    final_val_loss = float(h["val_loss"].iloc[-1])

    return {
        "best_epoch": best_epoch,
        "stop_epoch": stop_epoch,
        "epochs_after_best": stop_epoch - best_epoch,
        "best_val_loss": best_val_loss,
        "train_loss_at_best": train_loss_at_best,
        "gap_at_best": best_val_loss - train_loss_at_best,
        "final_train_loss": final_train_loss,
        "final_val_loss": final_val_loss,
        "final_gap": final_val_loss - final_train_loss,
        "min_train_loss": float(h["train_loss"].min()),
        "min_val_loss": float(h["val_loss"].min()),
    }


def prediction_to_history_path(prediction_path):
    name = prediction_path.name

    suffixes = [
        "_train_val_predictions_embeddings.npz",
        "_train_val_cal_test_predictions_embeddings.npz",
        "_train_val_cal_predictions_embeddings.npz",
        "_train_val_test_predictions_embeddings.npz",
    ]

    for suffix in suffixes:
        if name.endswith(suffix):
            return prediction_path.with_name(name.replace(suffix, "_history.npz"))

    return prediction_path.with_name(name.replace("_predictions_embeddings.npz", "_history.npz"))



## Compute validation metrics for all models

Metrics are computed in:

- standardized space
- physical space

In [7]:
metric_rows = []
quantile_rows = []

for model_id, path in prediction_files.items():
    if not path.exists():
        print(f"Skipping missing file: {model_id} -> {path}")
        continue

    data = np.load(path, allow_pickle=True)

    pred_val = data["pred_val"]
    y_val = data["y_val"]

    y_mean = data["y_mean"]
    y_std = data["y_std"]
    label_names = get_label_names(data)

    # -------------------------
    # Standardized space
    # -------------------------
    metrics_std = regression_metrics(y_val, pred_val)

    metric_rows.append({
        "model_id": model_id,
        "space": "standardized",
        "label": "global",
        "MSE": metrics_std["global_mse"],
        "RMSE": metrics_std["global_rmse"],
        "MAE": metrics_std["global_mae"],
        "Bias": np.nan,
        "Residual std": np.nan,
        "R2": np.nan,
    })

    for j, label in enumerate(label_names):
        metric_rows.append({
            "model_id": model_id,
            "space": "standardized",
            "label": label,
            "MSE": metrics_std["mse"][j],
            "RMSE": metrics_std["rmse"][j],
            "MAE": metrics_std["mae"][j],
            "Bias": metrics_std["bias"][j],
            "Residual std": metrics_std["residual_std"][j],
            "R2": metrics_std["r2"][j],
        })

    quantile_rows.extend(
        abs_error_quantiles(
            y_true=y_val,
            y_pred=pred_val,
            label_names=label_names,
            model_id=model_id,
            space="standardized",
        )
    )

    # -------------------------
    # Physical space
    # -------------------------
    y_val_phys = y_val * y_std + y_mean
    pred_val_phys = pred_val * y_std + y_mean

    metrics_phys = regression_metrics(y_val_phys, pred_val_phys)

    metric_rows.append({
        "model_id": model_id,
        "space": "physical",
        "label": "global",
        "MSE": metrics_phys["global_mse"],
        "RMSE": metrics_phys["global_rmse"],
        "MAE": metrics_phys["global_mae"],
        "Bias": np.nan,
        "Residual std": np.nan,
        "R2": np.nan,
    })

    for j, label in enumerate(label_names):
        metric_rows.append({
            "model_id": model_id,
            "space": "physical",
            "label": label,
            "MSE": metrics_phys["mse"][j],
            "RMSE": metrics_phys["rmse"][j],
            "MAE": metrics_phys["mae"][j],
            "Bias": metrics_phys["bias"][j],
            "Residual std": metrics_phys["residual_std"][j],
            "R2": metrics_phys["r2"][j],
        })

    quantile_rows.extend(
        abs_error_quantiles(
            y_true=y_val_phys,
            y_pred=pred_val_phys,
            label_names=label_names,
            model_id=model_id,
            space="physical",
        )
    )

summary_df = pd.DataFrame(metric_rows)
quantiles_df = pd.DataFrame(quantile_rows)

summary_df

,model_id,space,label,MSE,RMSE,MAE,Bias,Residual std,R2
0,M00_baseline_emb64_seed123,standardized,global,0.177084,0.420813,0.305378,NaN,NaN,NaN
1,M00_baseline_emb64_seed123,standardized,chirp_mass,0.140808,0.375244,0.268875,-0.013185,0.375013,0.857300
2,M00_baseline_emb64_seed123,standardized,total_mass,0.112468,0.335363,0.249743,-0.009372,0.335232,0.885663
3,M00_baseline_emb64_seed123,standardized,chi_eff,0.277973,0.527231,0.397517,-0.007467,0.527178,0.725214
4,M00_baseline_emb64_seed123,physical,global,58.075588,7.620734,4.431233,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
59,M06_WideCNN_emb128_pool4,standardized,chi_eff,0.283660,0.532597,0.405834,-0.004616,0.532578,0.719592
60,M06_WideCNN_emb128_pool4,physical,global,59.496357,7.713388,4.506614,NaN,NaN,NaN
61,M06_WideCNN_emb128_pool4,physical,chirp_mass,39.349358,6.272907,4.527555,0.317423,6.264858,0.854007
62,M06_WideCNN_emb128_pool4,physical,total_mass,139.084747,11.793420,8.813968,0.472019,11.783961,0.882820


## Global validation metrics

In [8]:
summary_df.query("space == 'standardized' and label == 'global'").sort_values("MSE")

,model_id,space,label,MSE,RMSE,MAE,Bias,Residual std,R2
8,M00_baseline_emb64_seed124,standardized,global,0.173310,0.416305,0.303117,NaN,NaN,NaN
16,M00_baseline_emb64_seed125,standardized,global,0.173485,0.416515,0.302875,NaN,NaN,NaN
40,M04_pooldeep_emb128_pool4,standardized,global,0.176160,0.419714,0.305510,NaN,NaN,NaN
0,M00_baseline_emb64_seed123,standardized,global,0.177084,0.420813,0.305378,NaN,NaN,NaN
24,M01_pool1_emb128,standardized,global,0.178520,0.422516,0.310376,NaN,NaN,NaN
56,M06_WideCNN_emb128_pool4,standardized,global,0.180994,0.425434,0.311171,NaN,NaN,NaN
48,M04_pooldeep_emb128_pool4_drop_wd,standardized,global,0.182728,0.427467,0.312132,NaN,NaN,NaN
32,M02_pool4_emb128,standardized,global,0.182914,0.427684,0.315345,NaN,NaN,NaN


## Per-label validation metrics: standardized space

In [9]:
summary_df.query("space == 'standardized' and label != 'global'").sort_values(["label", "MSE"])

,model_id,space,label,MSE,RMSE,MAE,Bias,Residual std,R2
19,M00_baseline_emb64_seed125,standardized,chi_eff,0.271617,0.521169,0.394862,0.009106,0.521089,0.731497
11,M00_baseline_emb64_seed124,standardized,chi_eff,0.275001,0.524405,0.397182,0.017485,0.524114,0.728152
3,M00_baseline_emb64_seed123,standardized,chi_eff,0.277973,0.527231,0.397517,-0.007467,0.527178,0.725214
27,M01_pool1_emb128,standardized,chi_eff,0.279148,0.528344,0.401955,-0.000254,0.528344,0.724053
43,M04_pooldeep_emb128_pool4,standardized,chi_eff,0.279810,0.528971,0.400771,-0.038120,0.527596,0.723398
59,M06_WideCNN_emb128_pool4,standardized,chi_eff,0.283660,0.532597,0.405834,-0.004616,0.532578,0.719592
35,M02_pool4_emb128,standardized,chi_eff,0.283983,0.532901,0.406288,-0.014028,0.532716,0.719272
51,M04_pooldeep_emb128_pool4_drop_wd,standardized,chi_eff,0.291615,0.540014,0.412341,-0.023142,0.539518,0.711728
9,M00_baseline_emb64_seed124,standardized,chirp_mass,0.136253,0.369125,0.266263,0.003366,0.369111,0.861916
41,M04_pooldeep_emb128_pool4,standardized,chirp_mass,0.137081,0.370245,0.264898,-0.019893,0.369710,0.861077


## Per-label validation metrics: physical space

In [ ]:
summary_df.query("space == 'physical' and label != 'global'").sort_values(["label", "RMSE"])

## Absolute-error quantiles

These are useful for checking whether a model improves only the mean error or also the tails.

In [ ]:
quantiles_df.query("space == 'physical'").sort_values(["label", "q90_abs_error"])

## Model files, histories, and ranking

This section links prediction files with training histories, summarizes convergence, and builds compact architecture-ranking tables.

## Build model file table

This table links each model ID with its prediction file and, when available, its training history file.

In [ ]:
model_file_rows = []

for model_id, pred_path in prediction_files.items():
    hist_path = prediction_to_history_path(pred_path)

    model_file_rows.append({
        "model_id": model_id,
        "prediction_file": pred_path,
        "prediction_exists": pred_path.exists(),
        "history_file": hist_path,
        "history_exists": hist_path.exists(),
    })

model_files_df = pd.DataFrame(model_file_rows)
model_files_df

## Compute training-history summaries

In [ ]:
history_rows = []

for row in model_files_df.itertuples(index=False):
    if not row.history_exists:
        continue

    history_df = load_history_npz(row.history_file)
    summary = summarize_history(history_df)
    summary["model_id"] = row.model_id
    history_rows.append(summary)

history_summary_df = pd.DataFrame(history_rows)

if not history_summary_df.empty:
    cols = ["model_id"] + [c for c in history_summary_df.columns if c != "model_id"]
    history_summary_df = history_summary_df[cols]

history_summary_df

## Architecture ranking

This table combines global validation MSE, selected physical error quantiles, and training-history information.

In [ ]:
global_rank = (
    summary_df
    .query("space == 'standardized' and label == 'global'")
    .loc[:, ["model_id", "MSE", "RMSE", "MAE"]]
    .rename(columns={
        "MSE": "val_MSE_global",
        "RMSE": "val_RMSE_global",
        "MAE": "val_MAE_global",
    })
)

phys_rmse = (
    summary_df
    .query("space == 'physical' and label != 'global'")
    .pivot(index="model_id", columns="label", values="RMSE")
    .add_prefix("phys_RMSE_")
    .reset_index()
)

phys_bias = (
    summary_df
    .query("space == 'physical' and label != 'global'")
    .pivot(index="model_id", columns="label", values="Bias")
    .add_prefix("phys_bias_")
    .reset_index()
)

phys_q90 = (
    quantiles_df
    .query("space == 'physical'")
    .pivot(index="model_id", columns="label", values="q90_abs_error")
    .add_prefix("phys_q90_abs_")
    .reset_index()
)

ranking_df = (
    global_rank
    .merge(phys_rmse, on="model_id", how="left")
    .merge(phys_bias, on="model_id", how="left")
    .merge(phys_q90, on="model_id", how="left")
)

if not history_summary_df.empty:
    ranking_df = ranking_df.merge(history_summary_df, on="model_id", how="left")

ranking_df = ranking_df.sort_values("val_MSE_global")
ranking_df

## Baseline seed variability

This section estimates how much the M00 baseline varies across random seeds.

In [ ]:
baseline_seed_ids = [
    "M00_baseline_emb64_seed123",
    "M00_baseline_emb64_seed124",
    "M00_baseline_emb64_seed125",
]

baseline_seed_df = ranking_df[ranking_df["model_id"].isin(baseline_seed_ids)].copy()
baseline_seed_df = baseline_seed_df.sort_values("model_id")
baseline_seed_df

In [ ]:
baseline_stats = (
    baseline_seed_df[[
        "val_MSE_global",
        "phys_RMSE_chirp_mass",
        "phys_RMSE_total_mass",
        "phys_RMSE_chi_eff",
    ]]
    .agg(["mean", "std", "min", "max"])
)

baseline_stats

## Candidate comparison against baseline distribution

In [ ]:
baseline_mean = baseline_stats.loc["mean", "val_MSE_global"]
baseline_std = baseline_stats.loc["std", "val_MSE_global"]

candidate_df = ranking_df.copy()
candidate_df["delta_vs_M00_seed_mean"] = candidate_df["val_MSE_global"] - baseline_mean
if pd.isna(baseline_std) or baseline_std == 0:
    candidate_df["z_vs_M00_seed_mean"] = np.nan
else:
    candidate_df["z_vs_M00_seed_mean"] = candidate_df["delta_vs_M00_seed_mean"] / baseline_std

candidate_df.sort_values("val_MSE_global")

## Save comparison tables

In [ ]:
comparison_dir = MODEL_RESULTS_DIR / "evaluation"
comparison_dir.mkdir(parents=True, exist_ok=True)

summary_csv = comparison_dir / "architecture_search_val_metrics.csv"
quantiles_csv = comparison_dir / "architecture_search_val_abs_error_quantiles.csv"
ranking_csv = comparison_dir / "architecture_search_ranking.csv"

summary_df.to_csv(summary_csv, index=False)
quantiles_df.to_csv(quantiles_csv, index=False)
ranking_df.to_csv(ranking_csv, index=False)

print("Saved:", summary_csv)
print("Saved:", quantiles_csv)
print("Saved:", ranking_csv)

## Detailed diagnostics for one selected model

Use this section to inspect residuals, prediction-vs-truth plots, SNR dependence, and embedding behavior for one model.

Do not run detailed plots for every model unless the summary tables justify it.

In [ ]:
SELECTED_MODEL_ID = "M00_baseline_emb64_seed123"
# SELECTED_MODEL_ID = "M01_pool1_emb128"
# SELECTED_MODEL_ID = "M02_pool4_emb128"
# SELECTED_MODEL_ID = "M04_pooldeep_emb128_pool4"

selected_path = prediction_files[SELECTED_MODEL_ID]

data = np.load(selected_path, allow_pickle=True)

pred_train = data["pred_train"]
y_train = data["y_train"]
emb_train = data["emb_train"]

pred_val = data["pred_val"]
y_val = data["y_val"]
emb_val = data["emb_val"]

y_mean = data["y_mean"]
y_std = data["y_std"]
label_names = get_label_names(data)

# Robust index loading across script versions.
train_idx = data["idx_train"] if "idx_train" in data.files else data["train_idx"]
val_idx = data["idx_val"] if "idx_val" in data.files else data["val_idx"]

pred_val_phys = pred_val * y_std + y_mean
y_val_phys = y_val * y_std + y_mean
residual_val_phys = y_val_phys - pred_val_phys

print("Selected model:", SELECTED_MODEL_ID)
print("prediction file:", selected_path)
print("label_names:", label_names)
print("pred_val:", pred_val.shape)
print("y_val:", y_val.shape)
print("emb_val:", emb_val.shape)
print("val_idx:", val_idx.shape)

In [ ]:
for j, label in enumerate(label_names):
    true = y_val_phys[:, j]
    pred = pred_val_phys[:, j]

    plt.figure(figsize=(8, 6))
    hb = plt.hexbin(true, pred, gridsize=70, mincnt=1, bins="log")
    plt.colorbar(hb, label="log10(count)")

    lo = min(true.min(), pred.min())
    hi = max(true.max(), pred.max())
    plt.plot([lo, hi], [lo, hi], linestyle="--")

    plt.xlabel(f"True {label}")
    plt.ylabel(f"Predicted {label}")
    plt.title(f"{SELECTED_MODEL_ID}: predicted vs true ({label})")
    plt.grid(True)
    plt.show()

In [ ]:
for j, label in enumerate(label_names):
    true = y_val_phys[:, j]
    residual = residual_val_phys[:, j]

    plt.figure(figsize=(6, 4.5))
    hb = plt.hexbin(true, residual, gridsize=70, mincnt=1, bins="log")
    plt.colorbar(hb, label="log10(count)")

    plt.axhline(0.0, linestyle="--")
    plt.xlabel(f"True {label}")
    plt.ylabel("Residual: true - predicted")
    plt.title(f"{SELECTED_MODEL_ID}: residual vs true ({label})")
    plt.grid(True)
    plt.show()

## Comparison between models

In [ ]:
for label_idx, label in enumerate(label_names):
    plt.figure(figsize=(7, 4))

    for model_id, path in prediction_files.items():
        if not path.exists():
            continue

        data = np.load(path, allow_pickle=True)
        pred_val = data["pred_val"]
        y_val = data["y_val"]
        y_mean = data["y_mean"]
        y_std = data["y_std"]

        pred_phys = pred_val * y_std + y_mean
        y_phys = y_val * y_std + y_mean
        residual = y_phys[:, label_idx] - pred_phys[:, label_idx]

        plt.hist(
            residual,
            bins=100,
            density=True,
            histtype="step",
            linewidth=1.5,
            label=model_id,
        )

    plt.axvline(0.0, linestyle="--")
    plt.xlabel(f"Residual: true - predicted ({label})")
    plt.ylabel("Density")
    plt.title(f"Validation residual comparison ({label})")
    plt.legend()
    plt.grid(True)
    plt.show()

## Training curves

In [ ]:
SELECTED_MODEL_ID = "M04_pooldeep_emb128_pool4_drop_wd"

selected = model_files_df.query("model_id == @SELECTED_MODEL_ID")

if len(selected) == 0:
    raise ValueError(
        f"Model not found: {SELECTED_MODEL_ID}. "
        f"Available models: {model_files_df['model_id'].tolist()}"
    )

selected = selected.iloc[0]

if not selected["history_exists"]:
    raise FileNotFoundError(f"No history file found for {SELECTED_MODEL_ID}")

history_df = load_history_npz(selected["history_path"])

best_idx = history_df["val_loss"].astype(float).idxmin()
best_epoch = int(history_df.loc[best_idx, "epoch"])
best_val = float(history_df.loc[best_idx, "val_loss"])
train_at_best = float(history_df.loc[best_idx, "train_loss"])

plt.figure(figsize=(7, 4.5))
plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
plt.axvline(best_epoch, linestyle="--", label=f"best epoch = {best_epoch}")
plt.scatter([best_epoch], [best_val], zorder=3)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"Training curve: {SELECTED_MODEL_ID}")
plt.legend()
plt.grid(True)
plt.show()

print(f"Best epoch:        {best_epoch}")
print(f"Best val loss:     {best_val:.6f}")
print(f"Train loss @ best: {train_at_best:.6f}")
print(f"Gap @ best:        {best_val - train_at_best:.6f}")
print(f"Stop epoch:        {int(history_df['epoch'].iloc[-1])}")

In [ ]:
MODELS_TO_PLOT = [
    "M00_baseline_emb64_seed123",
    "M00_baseline_emb64_seed124",
    "M00_baseline_emb64_seed125",
    "M01_pool1_emb128",
    "M02_pool4_emb128",
    "M04_pooldeep_emb128_pool4",
    "M04_pooldeep_emb128_pool4_drop_wd",
    "M06_WideCNN_emb128_pool4",
]

available_model_ids = set(model_files_df["model_id"].tolist())

missing = [m for m in MODELS_TO_PLOT if m not in available_model_ids]

if missing:
    print("These requested models were not found:")
    for m in missing:
        print("  ", repr(m))

    print("\nAvailable model_ids are:")
    for m in sorted(available_model_ids):
        print("  ", repr(m))

plt.figure(figsize=(9, 5))

for model_id in MODELS_TO_PLOT:
    selected = model_files_df.query("model_id == @model_id")

    if len(selected) == 0:
        continue

    selected = selected.iloc[0]

    if not selected["history_exists"]:
        print(f"Skipping {model_id}: no history found")
        continue

    h = load_history_npz(selected["history_path"])

    plt.plot(h["epoch"], h["val_loss"], label=model_id)

plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Validation loss comparison")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

for model_id in MODELS_TO_PLOT:
    selected = model_files_df.query("model_id == @model_id")

    if len(selected) == 0:
        continue

    selected = selected.iloc[0]

    if not selected["history_exists"]:
        continue

    h = load_history_npz(selected["history_path"])

    gap = h["val_loss"].astype(float) - h["train_loss"].astype(float)

    plt.plot(h["epoch"], gap, label=model_id)

plt.axhline(0.0, linestyle="--")
plt.xlabel("Epoch")
plt.ylabel("val_loss - train_loss")
plt.title("Generalization gap comparison")
plt.legend()
plt.grid(True)
plt.show()


# Final evaluation: 500k baseline

This section evaluates the final 500k SimpleCNN baseline using the real train/val/cal/test split. The main reported numbers should come from the test split.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_ROOT_500K = Path("/data/vserrano/cbc_pe_data")
RESULTS_500K = DATA_ROOT_500K / "results" / "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000"

MODEL_500K_ID = "500k_M00_baseline_emb64_seed123"

prediction_file_500k = RESULTS_500K / (
    "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000"
    "_SimpleCNN_Baseline_simple_emb64_mse_MSELoss_seed123"
    "_train_val_cal_test_predictions_embeddings.npz"
)

print("prediction_file_500k:", prediction_file_500k)
print("exists:", prediction_file_500k.exists())

assert prediction_file_500k.exists(), prediction_file_500k

prediction_file_500k: /data/vserrano/cbc_pe_data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_SimpleCNN_Baseline_simple_emb64_mse_MSELoss_seed123_train_val_cal_test_predictions_embeddings.npz
exists: True


In [2]:
data_500k = np.load(prediction_file_500k, allow_pickle=True)

print("Available keys:")
for k in data_500k.files:
    print(" ", k)

label_names = [str(x) for x in data_500k["label_names"].tolist()]
y_mean = data_500k["y_mean"]
y_std = data_500k["y_std"]

print("label_names:", label_names)
print("y_mean:", y_mean)
print("y_std:", y_std)

available_splits = [
    split for split in ["train", "val", "cal", "test"]
    if f"pred_{split}" in data_500k.files and f"y_{split}" in data_500k.files
]

print("available_splits:", available_splits)

for split in available_splits:
    print(
        split,
        "pred:", data_500k[f"pred_{split}"].shape,
        "y:", data_500k[f"y_{split}"].shape,
    )

Available keys:
  y_mean
  y_std
  label_names
  checkpoint_file
  dataset_path
  split_path
  label_stats_path
  model_config
  pred_train
  emb_train
  y_train
  idx_train
  pred_val
  emb_val
  y_val
  idx_val
  pred_cal
  emb_cal
  y_cal
  idx_cal
  pred_test
  emb_test
  y_test
  idx_test
  available_splits
label_names: ['chirp_mass', 'total_mass', 'chi_eff']
y_mean: [3.7452621e+01 9.4973969e+01 1.0226952e-03]
y_std: [16.484722   34.650166    0.44085518]
available_splits: ['train', 'val', 'cal', 'test']
train pred: (400000, 3) y: (400000, 3)
val pred: (40000, 3) y: (40000, 3)
cal pred: (30000, 3) y: (30000, 3)
test pred: (30000, 3) y: (30000, 3)


In [ ]:
def regression_metrics(y_true, y_pred):
    residual = y_true - y_pred

    mse = np.mean(residual**2, axis=0)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(residual), axis=0)
    bias = np.mean(residual, axis=0)
    residual_std = np.std(residual, axis=0)

    ss_res = np.sum(residual**2, axis=0)
    ss_tot = np.sum((y_true - np.mean(y_true, axis=0))**2, axis=0)
    r2 = 1.0 - ss_res / ss_tot

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "bias": bias,
        "residual_std": residual_std,
        "r2": r2,
        "global_mse": float(np.mean(residual**2)),
        "global_rmse": float(np.sqrt(np.mean(residual**2))),
        "global_mae": float(np.mean(np.abs(residual))),
    }


def add_metric_rows(rows, model_id, split, space, label_names, y_true, y_pred):
    m = regression_metrics(y_true, y_pred)

    rows.append({
        "model_id": model_id,
        "split": split,
        "space": space,
        "label": "global",
        "MSE": m["global_mse"],
        "RMSE": m["global_rmse"],
        "MAE": m["global_mae"],
        "Bias": np.nan,
        "Residual std": np.nan,
        "R2": np.nan,
        "n_samples": y_true.shape[0],
    })

    for j, label in enumerate(label_names):
        rows.append({
            "model_id": model_id,
            "split": split,
            "space": space,
            "label": label,
            "MSE": m["mse"][j],
            "RMSE": m["rmse"][j],
            "MAE": m["mae"][j],
            "Bias": m["bias"][j],
            "Residual std": m["residual_std"][j],
            "R2": m["r2"][j],
            "n_samples": y_true.shape[0],
        })


def add_quantile_rows(rows, model_id, split, space, label_names, y_true, y_pred):
    abs_err = np.abs(y_true - y_pred)

    for j, label in enumerate(label_names):
        rows.append({
            "model_id": model_id,
            "split": split,
            "space": space,
            "label": label,
            "q50_abs_error": np.quantile(abs_err[:, j], 0.50),
            "q68_abs_error": np.quantile(abs_err[:, j], 0.68),
            "q90_abs_error": np.quantile(abs_err[:, j], 0.90),
            "q95_abs_error": np.quantile(abs_err[:, j], 0.95),
            "q99_abs_error": np.quantile(abs_err[:, j], 0.99),
            "n_samples": y_true.shape[0],
        })

In [ ]:
metric_rows = []
quantile_rows = []

for split in available_splits:
    pred_std = data_500k[f"pred_{split}"]
    y_std_true = data_500k[f"y_{split}"]

    # Standardized space
    add_metric_rows(
        rows=metric_rows,
        model_id=MODEL_500K_ID,
        split=split,
        space="standardized",
        label_names=label_names,
        y_true=y_std_true,
        y_pred=pred_std,
    )

    add_quantile_rows(
        rows=quantile_rows,
        model_id=MODEL_500K_ID,
        split=split,
        space="standardized",
        label_names=label_names,
        y_true=y_std_true,
        y_pred=pred_std,
    )

    # Physical space
    y_phys = y_std_true * y_std + y_mean
    pred_phys = pred_std * y_std + y_mean

    add_metric_rows(
        rows=metric_rows,
        model_id=MODEL_500K_ID,
        split=split,
        space="physical",
        label_names=label_names,
        y_true=y_phys,
        y_pred=pred_phys,
    )

    add_quantile_rows(
        rows=quantile_rows,
        model_id=MODEL_500K_ID,
        split=split,
        space="physical",
        label_names=label_names,
        y_true=y_phys,
        y_pred=pred_phys,
    )

final_500k_summary_df = pd.DataFrame(metric_rows)
final_500k_quantiles_df = pd.DataFrame(quantile_rows)

final_500k_summary_df

In [ ]:
final_500k_summary_df.query(
    "space == 'standardized' and label == 'global'"
).sort_values("split")

In [ ]:
final_500k_summary_df.query(
    "space == 'physical' and label != 'global'"
).sort_values(["label", "split"])

In [ ]:
final_500k_quantiles_df.query(
    "space == 'physical'"
).sort_values(["label", "split"])

In [ ]:
test_metrics_physical = final_500k_summary_df.query(
    "split == 'test' and space == 'physical' and label != 'global'"
).copy()

test_metrics_physical

In [ ]:
test_metrics_std = final_500k_summary_df.query(
    "split == 'test' and space == 'standardized'"
).copy()

test_metrics_std

In [ ]:
stability_df = final_500k_summary_df.query(
    "space == 'standardized' and label != 'global' and split in ['val', 'cal', 'test']"
).pivot_table(
    index="label",
    columns="split",
    values="MSE",
)

stability_df["max_minus_min"] = stability_df.max(axis=1) - stability_df.min(axis=1)
stability_df["relative_spread_%"] = 100.0 * stability_df["max_minus_min"] / stability_df[["val", "cal", "test"]].mean(axis=1)

stability_df

In [ ]:
output_dir = Path("outputs/final_500k_evaluation")
output_dir.mkdir(parents=True, exist_ok=True)

final_500k_summary_df.to_csv(output_dir / "final_500k_summary_metrics.csv", index=False)
final_500k_quantiles_df.to_csv(output_dir / "final_500k_abs_error_quantiles.csv", index=False)
test_metrics_physical.to_csv(output_dir / "final_500k_test_metrics_physical.csv", index=False)
test_metrics_std.to_csv(output_dir / "final_500k_test_metrics_standardized.csv", index=False)

print("Saved to:", output_dir.resolve())

In [ ]:
final_500k_summary_df.query("space == 'standardized' and label == 'global'")

In [ ]:
final_500k_summary_df.query("split == 'test' and space == 'standardized'")

In [ ]:
final_500k_summary_df.query("split == 'test' and space == 'physical' and label != 'global'")

In [ ]:
stability_df

The final 500k SimpleCNN baseline generalizes consistently across validation, calibration and test splits. The global standardized MSE is 0.1379 on validation, 0.1384 on calibration and 0.1383 on test. Per-label test performance shows strongest accuracy for total mass (R² = 0.913), followed by chirp mass (R² = 0.890), while chi_eff remains the most challenging target (R² = 0.782). The near-identical val/cal/test losses indicate that the final test result is stable and not driven by a favorable split.

In [ ]:
final_500k_summary_df.query(
    "split == 'test' and space == 'physical' and label != 'global'"
)

In [ ]:
final_500k_quantiles_df.query(
    "split == 'test' and space == 'physical'"
)

En el conjunto de test, el modelo 500k alcanza un error absoluto mediano de 2.77 M_sun en chirp_mass, 5.69 M_sun en total_mass y 0.120 en chi_eff. El 90% de los errores permanecen por debajo de 8.73 M_sun, 16.36 M_sun y 0.331, respectivamente. Las diferencias entre los errores medianos y los cuantiles altos indican la presencia de colas de error, especialmente relevantes para chi_eff.

El modelo 500k muestra una generalización estable en val/cal/test, con MSE global estandarizado ≈ 0.138 en los tres splits. En test, obtiene R² = 0.890 para chirp_mass, R² = 0.913 para total_mass y R² = 0.782 para chi_eff. Los errores absolutos medianos son 2.77 M_sun, 5.69 M_sun y 0.120, respectivamente. Los cuantiles altos muestran colas de error no despreciables, especialmente en chi_eff, lo que justifica el uso de intervalos conformal y análisis local por dificultad.